# LangChain implementation

In [1]:
import os
from langchain_openai import ChatOpenAI

In [2]:
# Openai llm
# key = "sk-aakjgfkjgkdjkfjskadjkljglej95i0g950k;vkl;skvl;skv;d-09v0--5t05gkleklgkdlklkdlkfdksnbbdbdfbdbb"
# llm = ChatOpenAI(api_key = key)

llm = ChatOpenAI()

In [ ]:
print(llm)

In [ ]:
# ******************************************
# 1) Simple LLM
# Description: Interfaces for language models
# *******************************************
def run(prompt):
    try:
    
        messages = [ ("system", "You are a helpful assistant good in giving information",),
                    ("human", prompt), ]
        
        response = llm.invoke(messages)
        # content = response
        # print(response.content)
    
    except Exception as e:
        response = "Exception from run()." + "\n" + str(e)
    
    return (response)

In [ ]:
# 0-shot prompt
prompt = "Is time travel really popssible. Give 3 key points"
response = run(prompt)

In [ ]:
print("Full Response")
print(response)

print()
# Contents
print("Contents ...")
print(response.content)
print()

# Token usage
print("Tokens Usage ...")
print(response.response_metadata["token_usage"])

In [ ]:
# 1-shot prompt
prompt = '''
Generate a short, enticing product description for a 'WhisFlow' smart air purifier, 
highlighting its quiet operation and intelligent air quality sensing.

Start with the text 'Introducing the WhisperFlow humidifier....'
'''
response = run(prompt)
print(response.content)

# --------------------------------------------------------
# 2) Parameterized templates / Placeholder-based templates
# --------------------------------------------------------

'''
Use LangChain PromptTemplate when:
    Building AI applications
    Chains/agents
    Reusable workflows
    Multi-step pipelines
    Production systems
    Multi-agent orchestration
'''

In [ ]:
# *************************************************
#   2.1) Component: Prompt Templates
#   Build templates to optimize LLM queries / prompts
# *************************************************

# from langchain.prompts import PromptTemplate # old version
from langchain_core.prompts import PromptTemplate # recent

# 1)
# topic = "temples and architecture" # gives the full answer

# topic = "" # asks user to complete response
topic = "exploring caves"

try:
    pt = PromptTemplate(input_variables=["topic"], 
                        template="what are the top 5 destinations for people who love {topic}?" )

    # print(pt)
    prompt = pt.format(topic=topic)
    
    # prompt = pt.format()
    # Python f formatting will not identify if 'topic' is missing

    print(prompt)
    response = run(prompt)
    print(response.content)

except Exception as e:
    print("Exception : " + str(e))

In [ ]:
# 2) 2 parameters
temp = "tell me a {p1} fact about {p2}"
pt = PromptTemplate.from_template(temp)

print(pt)

In [ ]:
prompt = pt.format(p1="fun",p2="comuters")
print(prompt)
print()

response = run(prompt)
print(response.content)

In [ ]:
# 3) 3 parameters
temp = "write a short note on {p1} and its impact on {p2} and {p3}"
pt = PromptTemplate.from_template(temp)
prompt = pt.format(p1="social media",p2="children",p3="senior citizens")

print(prompt)
response = run(prompt)
print(response.content)


# ====================================================
# 3) Component: Chains
# Combine LLMs and Prompt templates to build workflows
# ====================================================

In [ ]:
# ***********************************************************************************************************
# example 3.1 
# simple chain without any input parameter and output parser
# # A basic single-step chain that takes a user query, formats it using a prompt, and gets an answer from the LLM.
# ***********************************************************************************************************

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# i) Create the LLM
# llm = ChatOpenAI()

# ii) Create the Prompt
qry = ''' 
    A company is planning to conduct a training program on AI to its leaders.
    Suggest a suitable program name in about 5 words.
    '''
prompt = PromptTemplate.from_template(qry)
print(prompt)

In [ ]:
# iii) Create the LLM Chain
# new method
# this method is called LCEL (LangChain Expression Language)
# Connect AI components such as prompts, models, data retrievers, and parsers using a simple, chainable syntax with the pipe | operator, enabling smooth data flow from one component to the next.

# It is designed to simplify the creation of complex AI pipelines while keeping code clean, modular, and readable.

chain = prompt | llm

In [ ]:
# iv) Run it
result = chain.invoke({})
print(result.content)

# ***********************************************************************************************************
# example 3.2 
# simple chain with input parameter and an output parser
# A basic single-step chain that takes a user query, formats it using a prompt, 
# gets an answer from the LLM, formats the response
# ***********************************************************************************************************

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# i) Create the LLM
# llm = ChatOpenAI()

# ii) Create the Prompt
prompt = PromptTemplate(input_variables=["topic"], 
                        template="Explain in 3 bullet points the concept of {topic}" )

In [ ]:
# iii) Create the LLM Chain
chain = prompt | llm | StrOutputParser()

In [ ]:
# iv) Run it
topic = "Carbon Dating"

result = chain.invoke({"topic":topic})
print(result)

'''
Use PromptTemplate when 
-----------------------
    your prompt is basically one text block.
    you have a single instruction
    you do not need roles like system/human
    your prompt is just one plain string
    you are building a simple chain quickly

example usage:
    summarization
    classification
    rewriting
    extraction
    one-shot Q&A

Use ChatPromptTemplate when
---------------------------
    you want the prompt in chat-message format like system, human, assistant.
    you are using a chat model
    you want system instructions
    you want separate human / AI / system messages
    you want to include chat history later
    you want better structure and control

example usage:
    assistants
    conversation apps
    role-based prompting
    multi-message workflows
    memory/chat-history use cases
'''

# ********************************************
# example 3.3 
# Example of ChatPromptTemplate with Parser
# ********************************************


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# Step 1: Define the prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful customer support assistant."),
    ("human", "Customer issue: {issue}")
])

# Step 2: Initialize LLM
# llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Step 3: Create chain (new style)
chain = prompt | llm | StrOutputParser()

# Step 4: Invoke with input
response = chain.invoke({
    "issue": "My internet connection keeps disconnecting every 10 minutes."
})

# print(response.content)
print(response)

In [3]:
# **************************
# example 3.4 : JSON Parser
# **************************
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI

In [4]:
# Step 1: Define parser with expected schema
parser = JsonOutputParser()
format_instructions = parser.get_format_instructions()

# Step 2: Create prompt template
prompt = ChatPromptTemplate.from_template("""
You are an AI assistant that classifies support tickets.

Extract the following fields:
- category
- priority
- suggested_action

{format_instructions}

User Query:
{query}
""")

# Step 3: Initialize LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Step 4: Create chain
chain = prompt | llm | parser

# Step 5: Invoke
response = chain.invoke({
    "query": "My laptop is overheating and shutting down frequently.",
    "format_instructions": format_instructions
})

print(response)

{'category': 'Hardware Issue', 'priority': 'High', 'suggested_action': 'Check for dust in the cooling vents, ensure the fan is working properly, and consider using a cooling pad. If the issue persists, contact IT support for further assistance.'}


In [5]:
response

{'category': 'Hardware Issue',
 'priority': 'High',
 'suggested_action': 'Check for dust in the cooling vents, ensure the fan is working properly, and consider using a cooling pad. If the issue persists, contact IT support for further assistance.'}

In [ ]:
# get the individual output key-values
cat = response['category']
priority = response['priority']
action = response['suggested_action']

print(action[:50])

In [ ]:
# *****************************
# example 3.5: Sequential Chain
# Sequential chains allow you to connect multiple chains and compose them into pipelines 
# that execute some specific scenario.
# Returns only the final response 
# *******************************

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# Step 1: Content
content_prompt = PromptTemplate( 
                input_variables=["title"],
                template="""You are an expert content writer.
                    Given a title, create a professional and high quality content with a suitable heading.
                    The content should not exceed 1000 words. 
                
                    Title: {title}
                
                    Content:""" 
                )

# Step 2: Summary
summary_prompt = PromptTemplate(
            input_variables=["contents"],
            template="""You are an expert editor of text documents.
                    Given the content on a topic, write a crisp 75-word summary. Retain the same heading.

                    Topic Content:
                    {contents}
                
                    Summary:"""
                )

parser = StrOutputParser()

In [ ]:
# runnable chains
content_chain = content_prompt | llm | parser
summary_chain = summary_prompt | llm | parser

# full pipeline
chain = content_chain | (lambda contents: {"contents": contents}) | summary_chain

In [ ]:
qry = "Importance of History subject in schools"

response = chain.invoke({"title": qry})
response

In [ ]:
# ************* To return intermediate results *****************


# Method 1 - using 2 LLM calls (not an effective method)
# ------------------------------------------------------

from langchain_core.runnables import RunnableLambda

content_chain = content_prompt | llm | parser
summary_chain = summary_prompt | llm | parser

def workflow(inputs):
    contents = content_chain.invoke({"title": inputs["title"]})
    summary = summary_chain.invoke({"contents": contents})
    
    return {
        "title": inputs["title"],
        "contents": contents,
        "summary": summary
    }

chain = RunnableLambda(workflow)

response = chain.invoke({"title": "Social media impact on children"})

print("CONTENT:\n", response["contents"])
print("\nSUMMARY:\n", response["summary"])

In [ ]:
# Method 2 - using 1 LLM call (more effective method)
# ------------------------------------------------------

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

combined_prompt = ChatPromptTemplate.from_template("""
You are a content writer.

For the given title:
1. Write detailed content.
2. Then write a short summary of that content.

Return the output strictly as JSON with these keys:
- contents
- summary

Title: {title}
""")

parser = JsonOutputParser()

chain = combined_prompt | llm | parser

response = chain.invoke({ "title": "Social media impact on children" })

In [ ]:
print(len(response))
print(response.keys())

In [ ]:
# print the actual value of the key
print(response['contents'])

In [ ]:
print(response['summary'])

# =========================================================
# 4) Component: Tools and Agents
# Agent: Use LLMs to choose a specific activity to perform
# Tool: Used by agents to perform a specific task
# ========================================================

In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

In [ ]:
# Tool 1
@tool
def convert_degree_to_fahrenheit(celsius: float) -> float:
    """
    Convert temperature from Celsius to Fahrenheit.
    """
    return (celsius * 9/5) + 32


# Tool 2
@tool
def convert_fahrenheit_to_degree(fahrenheit: float) -> float:
    """
    Convert temperature from Fahrenheit to Celsius.
    """
    return (fahrenheit - 32) * 5/9

In [ ]:
# Calling the tools directly
# Rare case

'''
c = 25
temp_f = convert_degree_to_fahrenheit.invoke({"celsius": c})
print(f"Celcius = {c}, Farenheit = {temp_f}")

f = 99
temp_c = convert_fahrenheit_to_degree.invoke({"fahrenheit": f})
print(f"Farenheit = {f}, Celcius = {temp_c}")
'''

In [ ]:
llm = ChatOpenAI(model="gpt-4o")

# Register the tools
tools = [ convert_degree_to_fahrenheit, convert_fahrenheit_to_degree ]

llm_with_tools = llm.bind_tools(tools)

In [ ]:
user_prompt = "convert 35 degrees C to F"

prompt = f''' You are an intelligent calculation agent who understands the input well and calls the correct tools.
             You have been given the following tools: {tools}

             Read the below user query well. 
             Select the right tool and automatically execute the user query using the tool name and parameters.
             Do not guess the tool name. If you cannot make out, return the response "No Tool Selected".

            User Prompt: {user_prompt}
'''

In [ ]:
response = llm_with_tools.invoke(prompt)

In [ ]:
tool_registry = {tool.name: tool for tool in tools }

tool_call = response.tool_calls
print(tool_call)

In [ ]:
tool_name = tool_call[0]["name"]
tool_args = tool_call[0]["args"]

print(tool_name,tool_args)

In [ ]:
result = tool_registry[tool_name].invoke(tool_args)

In [ ]:
print(result)

In [ ]:
# **************** NOTE ************************
# LLM decides the use of tools.
# if llm knows the answer, it will answer; else answer goes to the tool
# -----------------------------------------------------------------------------------

In [ ]:
# pip install -U langchain langchain-openai langchain-community wikipedia


from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langsmith import Client
from langchain.agents import create_agent
# from langchain.agents import create_react_agent, AgentExecutor

In [ ]:
%pip install wikipedia

In [ ]:
p = "What is climate change"
pt = PromptTemplate.from_template(p)
prompt = pt.format()

In [ ]:
# Tool
wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1000) )
tools = [wiki]

In [ ]:
agent = create_agent(model=llm, tools=tools, system_prompt="you are a helpful assistant")
content_msg = "What is climate change"
msg = {"messages": [{"role": "user", "content": content_msg}]}
response = agent.invoke(input=msg)

In [11]:
# (example 3) : 
# LLM + python functions as Tools
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langchain_classic.agents import create_react_agent, AgentExecutor
import nltk
from nltk import word_tokenize, pos_tag, ne_chunk

In [7]:
@tool
def extract_entities(text: str) -> dict:
    """Extract named entities and their types from the given input text."""
    entities = []

    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    tree = ne_chunk(tagged)

    for subtree in tree:
        if hasattr(subtree, "label"):
            for token, pos in subtree:
                entities.append(f"{subtree.label()}:{token}")

    return {"entities": entities}

In [8]:
# pip install duckduckgo-search

search = DuckDuckGoSearchRun()
tools = [search, extract_entities]

In [9]:
# ReAct Prompt — all 4 variables are mandatory
template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format strictly:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
Thought: I now know the final answer

Final Answer: Provide response in a JSON format as given below:
"text": <text>,  "entities": <list of extracted entities here> 

Begin!

Question: {input}
Thought:{agent_scratchpad}
"""

In [12]:
prompt = PromptTemplate.from_template(template)

# Create agent
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

In [13]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=False,handle_parsing_errors=True, 
                               max_iterations=5, max_execution_time=30, return_intermediate_steps=True)

In [14]:
user_content = ''' Generate a 200-word text document on some of the best tourist destinations around the world. 
Include entities like country, state, city, people, year, currency etc. wherever possible.
From the generated text document, extract 10 entities and entity types and give in a tabular format
'''

In [15]:
result = agent_executor.invoke({"input": user_content})

In [16]:
result.keys()

dict_keys(['input', 'output', 'intermediate_steps'])

In [17]:
print(result['input'])

 Generate a 200-word text document on some of the best tourist destinations around the world. 
Include entities like country, state, city, people, year, currency etc. wherever possible.
From the generated text document, extract 10 entities and entity types and give in a tabular format



In [18]:
output = result['output']

In [21]:
print(output)

```json
{
  "text": "The best travel destinations around the world for 2023, according to National Geographic, include iconic places like the Eiffel Tower in Paris, France. The Louvre, also in Paris, is another must-see. In the United States, the Grand Canyon in Arizona offers breathtaking views. Japan's Kyoto is renowned for its historic temples and cherry blossoms. For a tropical escape, the Maldives offers stunning beaches and luxury resorts. In Africa, the Serengeti in Tanzania is famous for its wildlife and the Great Migration. Australia's Great Barrier Reef is a paradise for divers. In South America, Machu Picchu in Peru is a testament to the Inca civilization. Italy's Rome is rich in history with landmarks like the Colosseum. Finally, the vibrant city of Rio de Janeiro in Brazil is known for its Carnival and beaches.",
  "entities": [
    {"entity": "National Geographic", "type": "ORGANIZATION"},
    {"entity": "Eiffel Tower", "type": "ORGANIZATION"},
    {"entity": "Paris", "ty

In [23]:
import re

In [24]:
output = re.sub("```", "",output)
output = re.sub("json\n","",output)
print(output)

{
  "text": "The best travel destinations around the world for 2023, according to National Geographic, include iconic places like the Eiffel Tower in Paris, France. The Louvre, also in Paris, is another must-see. In the United States, the Grand Canyon in Arizona offers breathtaking views. Japan's Kyoto is renowned for its historic temples and cherry blossoms. For a tropical escape, the Maldives offers stunning beaches and luxury resorts. In Africa, the Serengeti in Tanzania is famous for its wildlife and the Great Migration. Australia's Great Barrier Reef is a paradise for divers. In South America, Machu Picchu in Peru is a testament to the Inca civilization. Italy's Rome is rich in history with landmarks like the Colosseum. Finally, the vibrant city of Rio de Janeiro in Brazil is known for its Carnival and beaches.",
  "entities": [
    {"entity": "National Geographic", "type": "ORGANIZATION"},
    {"entity": "Eiffel Tower", "type": "ORGANIZATION"},
    {"entity": "Paris", "type": "GP

In [ ]:
import json

In [ ]:
type(output)

# Database as a Tool
'''
Flow:
1. User prompt
2. LLM selects one logical tool name
3. Python function executes matching SQL tool
4. LLM generates final analyst-style answer
'''

In [25]:
import os
import mysql.connector
import pandas as pd
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import pymysql

In [26]:
host = "localhost"; database="ey"; port=3306; user="root"; pwd="root"

# 1) Get MySQL Connection
# -----------------------
def get_mysql_conn():
    try:
        conn = mysql.connector.connect( host=host, port=port, database=database, user=user, password=pwd)
    except Exception as e:
        conn = str(e)

    return(conn)

In [ ]:
# Insert the response in the MySQL

def ExecuteQuery(action, query, values=None):
    ret = {"status": '', "message": "", "record": ""}
    act_msg = ''
    CURSOR = conn.cursor(dictionary=True)

    try:
        if action == "I":
            act_msg = "Inserted"
        elif action == "U":
            act_msg = "Updated"
        elif action == "D":
            act_msg = "Deleted"
        elif action == "S":
            act_msg = "Retrieved"

        if action in ["I", "U", "D"]:
            CURSOR.execute(query, values)
            conn.commit()
            ret["status"] = "SUCCESS"
            ret["message"] = f"{CURSOR.rowcount} Record(s) {act_msg}"
            ret["record"] = ''
        elif action == "S":
            if values is not None:
                CURSOR.execute(query, values)
            else:
                CURSOR.execute(query)

            data = CURSOR.fetchall()
            ret["status"] = "SUCCESS"
            ret["message"] = f"{len(data)} Record(s) {act_msg} "
            ret["record"] = pd.DataFrame(data)

    except Exception as e:
        ret["status"] = "EXCEPTION"
        ret["message"] = str(e)
        ret["record"] = ''
        CURSOR.close()

    finally:
        CURSOR.close()

    return (ret)

In [ ]:

# 3. SQL Tool Definitions
# -----------------------------
def user_access_tool():
    query = """
    SELECT * FROM data_security  WHERE failed_logins_24h > 3 AND department = 'Risk';
    """
    return ExecuteQuery(action="S",query=query)

def risk_threat_scoring_tool():
    query = """
    SELECT department, AVG(risk_score) AS avg_risk FROM data_security GROUP BY department ORDER BY avg_risk DESC;
    """
    return ExecuteQuery(action="S",query=query)

def data_exfiltration_tool():
    query = """
    SELECT user_id, device_type, data_sensitivity_level
    FROM data_security
    WHERE data_sensitivity_level > 4;
    """
    return ExecuteQuery(query=query,action="S")

def device_endpoint_compliance_tool():
    query = """
    SELECT device_type, encryption_in_transit_flag, unmanaged_device_flag
    FROM data_security
    WHERE unmanaged_device_flag = 0;
    """
    return ExecuteQuery(query=query,action="S")

def fallback_sql_tool():
    query = """ SELECT * FROM data_security LIMIT 10; """
    return ExecuteQuery(query=query,action="S")

In [ ]:
# ------------------- LLM -------------------
llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [ ]:

llm = ChatOpenAI(model="gpt-4o", temperature=0)

sys_prompt = """
You are a tool-selection agent for a BFSI data security dataset.

Choose exactly one tool from the following:

1. user_access
Use for:
- failed logins
- suspicious login behavior
- privileged users
- non-business hours access
- authentication issues

2. risk_threat_scoring
Use for:
- risk score
- threat score
- high-risk users
- department-wise risk
- average risk

3. data_exfiltration
Use for:
- DLP violations
- customer financial records
- data sensitivity
- data transfer
- data leakage
- exfiltration

4. device_endpoint_compliance
Use for:
- unmanaged devices
- endpoint compliance
- encryption
- non-compliant devices
- device security

Return only the tool name.
"""

In [ ]:
select_tool_prompt = ChatPromptTemplate.from_messages([ ("system", sys_prompt), ("human", "{user_prompt}") ])

select_tool_chain = select_tool_prompt | llm | StrOutputParser()

In [ ]:
user_prompt = "Show non-compliant endpoints connected to confidential banking resources." 

selected_tool = select_tool_chain.invoke({ "user_prompt": user_prompt})

selected_tool = selected_tool.strip().lower()

valid_tools = [ "user_access", "risk_threat_scoring", "data_exfiltration", "device_endpoint_compliance" ]

if selected_tool not in valid_tools:
    selected_tool = "fallback"

print(selected_tool)

In [ ]:
if selected_tool == "user_access":
    sql_result = user_access_tool()

elif selected_tool == "risk_threat_scoring":
    sql_result = risk_threat_scoring_tool()

elif selected_tool == "data_exfiltration":
    sql_result = data_exfiltration_tool()

elif selected_tool == "device_endpoint_compliance":
    sql_result = device_endpoint_compliance_tool()

else:
    sql_result = fallback_sql_tool()

status = sql_result
if status["status"] == "SUCCESS":
    print(status["record"])